# Use Case 5 — DECIDE pipeline: a business-rule decision flow with two chained real LLM stages

`agenttune.decide` is a separate, YAML-template-driven pipeline engine — not the agentic spine — meant for business decision flows like KYC triage. This notebook builds a 2-stage template (classify sentiment → decide an action) and runs it through `agenttune.api.run_pipeline` with a real model behind each stage, proving stage 2 genuinely reads stage 1's real output rather than each stage being independently canned.

In [1]:
import sys, os, tempfile
sys.path.insert(0, ".")
import yaml
import unittest.mock as mock
from _real_backends import real_generate, extract_json_object

from agenttune.api import arun_pipeline

print("Ready.")


Ready.


## 1. Define the template — a 2-stage decision pipeline

Stage 1 classifies sentiment; stage 2 decides an action FROM stage 1's output (`{s1.output.sentiment}` is interpolated into stage 2's prompt at run time).

In [2]:
def write_two_stage_template(tmpdir):
    template = {
        "id": "support/triage", "name": "Support Ticket Triage", "version": "1.0.0",
        "stages": [
            {"id": "s1", "type": "llm_call", "model": "gpt-4",
             "prompt": "Classify sentiment: {input_text}", "max_iterations": 1},
            {"id": "s2", "type": "llm_call", "model": "gpt-4",
             "prompt": "Stage2 Sentiment: {s1.output.sentiment}", "max_iterations": 1},
            {"id": "s3", "type": "output", "verdict_field": "s2.output.decision", "destinations": []},
        ],
        "edges": [
            {"from": "s1", "to": "s2", "condition": None},
            {"from": "s2", "to": "s3", "condition": None},
            {"from": "s3", "to": "__end__", "condition": None},
        ],
    }
    tdir = os.path.join(tmpdir, "templates", "support")
    os.makedirs(tdir, exist_ok=True)
    tpath = os.path.join(tdir, "triage.yaml")
    with open(tpath, "w") as f:
        yaml.dump(template, f)
    cpath = os.path.join(tmpdir, "config.yaml")
    with open(cpath, "w") as f:
        yaml.dump({"api_keys": {"openai": "unused-since-we-patch-litellm"}}, f)
    return tpath, cpath

print("template writer defined.")


template writer defined.


## 2. Wire the real model in behind `litellm.acompletion`

Each stage gets its own tight system prompt — stage 1 for sentiment, stage 2 for the escalate/close decision — both hitting the same real local model.

In [3]:
SENTIMENT_SYSTEM = (
    'You are a JSON-only API. Output exactly one JSON object {"sentiment": "positive"} '
    'or {"sentiment": "negative"} classifying the input text. No other output.\n\n'
    'Example\nInput: "I love this!"\nOutput: {"sentiment": "positive"}'
)
ACTION_SYSTEM = (
    'You are a JSON-only API. You are given a sentiment label of "positive" or "negative". '
    'If the sentiment is "positive", output {"decision": "close"}. '
    'If the sentiment is "negative", output {"decision": "escalate"}. '
    'Output ONLY the JSON object, nothing else.\n\n'
    'Example 1\nSentiment: positive\nOutput: {"decision": "close"}\n\n'
    'Example 2\nSentiment: negative\nOutput: {"decision": "escalate"}'
)

async def two_stage_acompletion(**kwargs):
    prompt = kwargs["messages"][-1]["content"]
    if "Stage2" in prompt:
        text = real_generate(prompt, system=ACTION_SYSTEM, max_new_tokens=15)
    else:
        text = real_generate(prompt, system=SENTIMENT_SYSTEM, max_new_tokens=15)
    return {"choices": [{"message": {"content": text}}]}

print("real litellm wiring defined.")


real litellm wiring defined.


## 3. Run it on a happy ticket and an angry ticket

(Using `arun_pipeline` with `await` — Jupyter's kernel already runs an event loop, so `run_pipeline`'s internal `asyncio.run()` can't be used directly inside a cell.)

In [4]:
with tempfile.TemporaryDirectory() as tmp:
    tpath, cpath = write_two_stage_template(tmp)
    with mock.patch("agenttune.decide.stages.base.litellm.acompletion", side_effect=two_stage_acompletion):
        happy = await arun_pipeline(tpath, "I am absolutely delighted with this service!", config=cpath)
        angry = await arun_pipeline(tpath, "This is infuriating and completely unacceptable.", config=cpath)

print(f"happy ticket -> verdict={happy.verdict!r}  is_complete={happy.is_complete}  error={happy.error}")
print(f"angry ticket -> verdict={angry.verdict!r}  is_complete={angry.is_complete}  error={angry.error}")

assert happy.verdict == "close"
assert angry.verdict == "escalate"
print("\nPASS — stage 2's real decision genuinely tracked stage 1's real sentiment output.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

happy ticket -> verdict='close'  is_complete=True  error=None
angry ticket -> verdict='escalate'  is_complete=True  error=None

PASS — stage 2's real decision genuinely tracked stage 1's real sentiment output.


---
**See also:** this pipeline engine is deliberately separate from the agentic spine used in the other notebooks — DECIDE is for config-driven business decision flows, the spine is for building and training agents. The two do share the self-healing closed-loop infrastructure. See the separate API-usage notebook in this folder for the full `agenttune.api`/CLI reference.